# Stage 4 — Baseline ASR (IndicConformer 600M)

**Attach:** `sarvam-diar-code`, `sarvam-diar-audio`, `sarvam-diar-stage3`.
**Settings:** GPU (T4) on, Internet on. No `HF_TOKEN` — the model is public.

This stage turns audio into **words with timestamps**, and nothing else. It never
sees a speaker label and never sees a diarization hypothesis. Attribution is a
separate CPU stage, so a single ASR run is reused across every diarization system
and every Stage 5 correction — which is what makes a cpWER delta attributable to
the labelling rather than to the ASR having been fed different audio.

### Why this is its own notebook

Whisper and IndicConformer want incompatible CUDA stacks. `onnxruntime-gpu`
installs its own `nvidia-cudnn-cu12`, which replaces the cuDNN that CTranslate2
(the runtime behind faster-whisper) was built against. The result is not an
error: Whisper silently falls back to CPU and a run that should take minutes
sits on an idle GPU for a quarter of an hour saying nothing.

Rather than fight that, each system gets its own session. They share nothing at
runtime — separate manifests, separate output directories, neither reads the
other — so the split costs nothing and removes a whole class of silent failure.
Save each notebook's output as a dataset; `stage4_attribute.py` is CPU-only and
attaches both.

### Why ONNX rather than NeMo

The `.nemo` checkpoint declares `tokenizer.type: multilingual` (stock NeMo
dispatches its aggregate tokenizer only on `agg`) and `multisoftmax: True` on
**both** the RNNT and CTC decoders, which upstream NeMo cannot instantiate. That
needs AI4Bharat's NeMo fork, which pins an older Python and torch than Kaggle
provides. The ONNX export bakes those fork features into the graph, so no fork is
required.

We take the **CTC branch**, not RNNT. The RNNT joint ships one output head per
language (`joint_post_net_<lang>.onnx`), so it would need a language decision per
clip — and the cheap source of that decision is the reference transcript's
script, which is ground truth leaking into the pipeline. The CTC head is a single
1024 → 5632 projection over the whole aggregate vocabulary. Because that
vocabulary is 22 per-language blocks concatenated in order, the argmax index
identifies the language for free.

### onnxruntime, carefully

Two traps, both of which land you silently on CPU:

- installing `onnxruntime-gpu` **alongside** the preinstalled `onnxruntime`
  leaves the CPU binaries in charge, so remove both first
- the latest `onnxruntime-gpu` (1.29) is built against CUDA 13 and dies with
  `libcublasLt.so.13: cannot open shared object file`. Kaggle ships CUDA 12

**Restart the kernel after this cell.** Do *not* add torch's NVIDIA libs to
`LD_LIBRARY_PATH` to force the provider — the version pin is what fixes it, and
the loader path breaks other CUDA consumers.

In [ ]:
!pip uninstall -y -q onnxruntime onnxruntime-gpu
!pip install -q "onnxruntime-gpu==1.20.2" librosa

In [ ]:
import onnxruntime as ort
print(ort.__version__, ort.get_available_providers())

`CUDAExecutionProvider` must appear above. Without it the run is hours instead of minutes.

In [ ]:
import pathlib, shutil

ROOT  = pathlib.Path("/kaggle/input")
CODE  = next(p.parent for p in ROOT.rglob("stage4_asr.py"))
AUDIO = next(p.parent for p in ROOT.rglob("*.wav"))
WORK  = pathlib.Path("/kaggle/working/data")
WORK.mkdir(parents=True, exist_ok=True)

for f in CODE.glob("*.py"):
    shutil.copy(f, "/kaggle/working/")

# Restore anything already produced -- Stage 3 RTTMs, and the other ASR system's
# words if its dataset is attached. Nothing here is required by this notebook;
# it is what lets stage4_attribute.py run later without re-attaching everything.
for src in sorted(ROOT.rglob("data")):
    if src.is_dir() and any((src / d).exists() for d in ("hyp", "ref", "asr")):
        shutil.copytree(src, WORK, dirs_exist_ok=True)
        print("restored", src)

print("CODE   :", CODE)
print("AUDIO  :", AUDIO, len(list(AUDIO.glob("*.wav"))), "wavs")
print("scripts:", sorted(p.name for p in pathlib.Path("/kaggle/working").glob("*.py")))
for sub in ("hyp", "asr"):
    d = WORK / sub
    if d.exists():
        for x in sorted(d.glob("*")):
            n = len(list(x.rglob("*.rttm"))) + len(list(x.rglob("*.json")))
            print(f"  {sub}/{x.name}: {n}")

### Is the script actually the current one?

Re-uploading `sarvam-diar-code` does not refresh `/kaggle/working`; the copy cell
above must run after the new dataset version is attached, which needs a session
restart. The assert below fails loudly instead of silently re-running the old
vocabulary mapping.

The wipe matters just as much. `Manifest.done()` skips any clip already recorded,
and the setup cell restores `data/asr/` from attached datasets — so words written
by the old mapping count as finished, and a resume would transcribe nothing while
reporting success. Set `WIPE = False` once a clean run exists.

In [ ]:
WIPE = True   # set False once a clean run exists and you want to resume it

import pathlib, shutil

src = pathlib.Path("/kaggle/working/stage4_asr.py").read_text(encoding="utf-8")
print("BLOCK_SIZE in the copied script:", src.count("BLOCK_SIZE"), "hits (expect 2)")
assert "toks = toks[:BLOCK_SIZE]" in src, (
    "stale stage4_asr.py -- re-upload sarvam-diar-code, restart the session, "
    "and re-run the setup cell above"
)

out = pathlib.Path("/kaggle/working/data/asr/indicconformer")
if WIPE and out.exists():
    n = len(list((out / "words").glob("*.json")))
    shutil.rmtree(out)
    print(f"wiped {n} clips of previous indicconformer output")
print("present now:", sorted(p.name for p in
      pathlib.Path("/kaggle/working/data/asr").glob("*")))

## Smoke test — 3 clips

- `onnxruntime providers:` contains **CUDAExecutionProvider**, no fallback warning
- `frontend: AI4Bharat TorchScript on cuda` — the fallback reimplementation is a
  last resort, and its output should be checked harder if it is used
- `vocab 5632 tokens over 22 languages, blank id 5632` — 5654 means the
  257-entry blocks were not trimmed to 256 and every index is off
- **RTF around 0.007.** Anything near 0.2 is CPU

In [ ]:
!python stage4_asr.py --system indicconformer --data data --wav-dir {AUDIO} --limit 3

### The check that actually matters

A clean summary line does not prove the decode is right. Read the `text` line:

- **it must be readable prose in ONE script, with no foreign characters at all.**
  On `0AEEA8NyVwY__000011000_000609000` the opening is
  `नमस्कार मी गौरव जोशी आणि मी अमोल कऱ्हाडकर …`. A single stray glyph means the
  language mask is not being applied
- **words that are phonetically right but spelled across several scripts**
  (`ನमस्कार`, `ଗౌरਵ`) is the multisoftmax failure. The CTC head was exported with
  `multisoftmax: True`: its softmax was trained over ONE language's 256-token
  block at a time, so logits from different blocks are on incomparable scales
  and a global argmax over all 5632 picks a different block nearly every frame.
  The decode must pick the clip's language from the frame votes and then argmax
  *within* that block. `--system indicconformer_free` reproduces the broken
  behaviour on purpose, as the ablation below
- **timestamps resetting every ~28 s** means the chunk offset is not applied, so
  every word after the first chunk is pinned to the wrong moment. WER would look
  fine and attribution would be destroyed
- **timestamps resetting every ~28 s** means the chunk offset is not applied, so
  every word after the first chunk is pinned to the wrong moment. WER would look
  fine and attribution would be destroyed

In [ ]:
import json, glob

files = sorted(glob.glob("/kaggle/working/data/asr/indicconformer/words/*.json"))
print(len(files), "clips transcribed")
d = json.load(open(files[0], encoding="utf-8"))
w = d["words"]
print("clip     :", d["clip_id"][:44], f'{d["duration"]:.1f}s')
print("lang     :", d.get("lang"), d.get("lang_counts", ""))
print("words    :", len(w))
print("first    :", w[:6])
print("last     :", w[-3:])
print("text     :", " ".join(x["w"] for x in w[:40]))
print("span     :", w[0]["start"], "->", w[-1]["end"], "of", d["duration"], "s")
print("monotonic:", all(a["start"] <= b["start"] for a, b in zip(w, w[1:])))
print("in bounds:", w[-1]["end"] <= d["duration"] + 1)

## Full run

Resumable: clips already marked `ok` are skipped, so a dead session restarts at the clip that was in flight. At RTF 0.007 the whole corpus is about five minutes.

In [ ]:
!python stage4_asr.py --system indicconformer --data data --wav-dir {AUDIO}

### Ablation — the same model with the language mask off

`indicconformer_free` is the identical checkpoint, identical features, identical
words; the only difference is that the argmax runs over all 5632 tokens instead
of the chosen language's 256. That is what a multisoftmax head does when you
decode it as if it were a single softmax, and it turns fluent Marathi into
`ನमस्कार ଗౌरਵ` — right sounds, six scripts.

Another five minutes of GPU buys a measured WER delta for the writeup instead of
an assertion, and it is the evidence that the language mask is a correctness fix
rather than a tuning choice.

In [ ]:
!python stage4_asr.py --system indicconformer_free --data data --wav-dir {AUDIO}

In [ ]:
import json, pathlib

mf = pathlib.Path("/kaggle/working/data/asr/indicconformer/manifest.jsonl")
recs = [json.loads(l) for l in mf.read_text(encoding="utf-8").splitlines() if l.strip()]
ok = [r for r in recs if r["status"] == "ok"]
print(f"indicconformer: {len(ok)} ok / {len(recs)} records, "
      f"{sum(r['n_words'] for r in ok):,} words")
for r in recs:
    if r["status"] != "ok":
        print("  fail:", r["clip_id"][:36], r.get("error", "")[:100])

rtfs = [r["rtf"] for r in ok if r.get("rtf")]
if rtfs:
    print(f"rtf: min {min(rtfs):.4f}  median {sorted(rtfs)[len(rtfs)//2]:.4f}  max {max(rtfs):.4f}")

langs = {}
for r in ok:
    langs[r.get("lang")] = langs.get(r.get("lang"), 0) + 1
print("languages:", dict(sorted(langs.items(), key=lambda kv: -kv[1])))

## Save

**Save Version → Quick Save**, then Output tab → **New dataset**, named
`sarvam-diar-asr-indic`. Notebook outputs re-point at the latest version, which
is how the Stage 3 RTTMs went missing; a dataset does not.

In [ ]:
!du -sh /kaggle/working/data/asr/* 2>/dev/null
!find /kaggle/working/data/asr -name "*.json" | wc -l